In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns


In [4]:
!nvidia-smi

Tue Oct 21 18:10:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.76.04              Driver Version: 580.97         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   56C    P8             10W /  115W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

See the original commpetition site for tips:    
https://www.kaggle.com/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/rules    
 

This notebook obtains the data here because the data format is more appropriate for TensorFlow  (especially memory efficiency)      
https://www.kaggle.com/datasets/sionehoghen/facial-expression/data  
But the results were not optimal, so an attempt with a another dataset from the same kaggle user is used for this 2nd attempt.  
Hopefully it is labeled better and that is the only cause of the problem:  
https://www.kaggle.com/datasets/sionehoghen/facial-expressions 


# ====================================================================  
### Load the data    
# ====================================================================  

In [7]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
import pathlib
from pathlib import Path

tf.random.set_seed(42)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
train_path=Path(r"data2/train")
train_paths=train_path.glob(r"*/*.jpg")
train_paths=list([str(path) for path in train_paths])
train_labels=[filename.split('/')[2] for filename in train_paths]
test_path=Path(r"data2/test")
test_paths=test_path.glob(r"*/*.jpg")
test_paths=list([str(path) for path in test_paths])
test_labels=[filename.split('/')[2] for filename in test_paths]

In [9]:
print(len(test_paths))
print(len(train_paths))
print(train_paths[0::10_000])
print(train_labels[0::10_000])
print(test_paths[::1_000])
print(test_labels[::1_000])

7178
28708
['data2/train/angry/Training_10118481.jpg', 'data2/train/happy/Training_27789087.jpg', 'data2/train/neutral/Training_87131839.jpg']
['angry', 'happy', 'neutral']
['data2/test/angry/PrivateTest_10131363.jpg', 'data2/test/disgust/PrivateTest_85928336.jpg', 'data2/test/fear/PublicTest_82742487.jpg', 'data2/test/happy/PublicTest_12933636.jpg', 'data2/test/neutral/PrivateTest_29667267.jpg', 'data2/test/neutral/PublicTest_86872627.jpg', 'data2/test/sad/PublicTest_51894157.jpg', 'data2/test/surprise/PublicTest_6010264.jpg']
['angry', 'disgust', 'fear', 'happy', 'neutral', 'neutral', 'sad', 'surprise']


# ========================================================  
# Encode
# ========================================================  

In [10]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
train_labels=le.fit_transform(train_labels)
test_labels=le.transform(test_labels)


In [11]:
locate_label_one=pd.DataFrame({'ones':train_labels})
indices=locate_label_one.loc[locate_label_one['ones']==1].index
indices

Index([3995, 3996, 3997, 3998, 3999, 4000, 4001, 4002, 4003, 4004,
       ...
       4421, 4422, 4423, 4424, 4425, 4426, 4427, 4428, 4429, 4430],
      dtype='int64', length=436)

In [12]:
train_paths=train_paths+list(train_paths[indices[0]:indices[-1]+1])*7
print(train_paths[-2:])
train_labels=list(train_labels)+list(train_labels[indices[0]:indices[-1]+1])*7

['data2/train/disgust/Training_99747227.jpg', 'data2/train/disgust/Training_99947220.jpg']


In [13]:
display(pd.DataFrame({'labels':train_labels}).value_counts())
display(pd.DataFrame({'labels':test_labels}).value_counts())

labels
3         7214
4         4965
5         4830
2         4097
0         3995
1         3488
6         3171
Name: count, dtype: int64

labels
3         1774
5         1247
4         1233
2         1024
0          958
6          831
1          111
Name: count, dtype: int64

In [14]:
train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

In [15]:
print(train_labels[:2])
print(test_labels[:2])

[[1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]]
[[1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0.]]


In [16]:
train_labels.shape

(31760, 7)

# calculate weights to input due to inbalanced labels

In [17]:
#compute weights
label_totals=train_labels.sum(axis=0)
label_weights=label_totals.max()/label_totals



label_weights={e: weight for e,weight in enumerate(label_weights)}
print(label_weights)

{0: np.float64(1.8057571964956196), 1: np.float64(2.0682339449541285), 2: np.float64(1.7608005857944837), 3: np.float64(1.0), 4: np.float64(1.4529707955689828), 5: np.float64(1.4935817805383023), 6: np.float64(2.2749921160517186)}


# ====================================================================  
### Preprocess      
# ==================================================================== 

In [18]:
img_size=96
channel=3  # not used consistently. Sometimes int(3) is used
batch_size=32

In [19]:
def load(image,label):
    """ accepts the image_path and one_hot encoded label"""
    image = tf.io.read_file(image) 
    image = tf.io.decode_jpeg(image,channels=channel) #decode to unit8 tensor   
    return image,label

In [20]:
#augmentation

resize = tf.keras.Sequential([
    tf.keras.layers.Resizing(img_size,img_size)
])
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(height_factor=(0.1,-0.05))
])


I0000 00:00:1761087748.307146    2581 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3584 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [21]:
#function to create a tensorflow object
batch_size=32
autotune=tf.data.AUTOTUNE
def get_dataset(paths,labels,train=True):
    image_paths     =tf.convert_to_tensor(paths)
    labels          =tf.convert_to_tensor(labels)

    image_dataset   = tf.data.Dataset.from_tensor_slices(image_paths)
    label_dataset   = tf.data.Dataset.from_tensor_slices(labels)

    dataset         = tf.data.Dataset.zip((image_dataset,label_dataset))

    dataset         = dataset.map(lambda image, label : load(image,label))  #call the load function defined earlier  
    dataset         = dataset.map(lambda image, label: (resize(image),label),num_parallel_calls=autotune) #call the resize layers
    dataset         = dataset.shuffle(1_000)
    dataset         = dataset.batch(batch_size)

    if train:
        dataset     = dataset.map(lambda image, label: (data_augmentation(image),label), num_parallel_calls=autotune)

    dataset         = dataset.repeat()

    return dataset


In [22]:
from sklearn.model_selection import train_test_split
train_paths_train,val_paths,train_labels_train,val_labels=train_test_split(train_paths,train_labels,test_size=0.25)

In [23]:
#create train dataset object and verify it
%time train_dataset = get_dataset(train_paths_train,train_labels_train)
%time val_dataset   = get_dataset(val_paths,val_labels,train=False)
image,label = next(iter(val_dataset))
print(image.shape)
print(label.shape)

CPU times: user 993 ms, sys: 144 ms, total: 1.14 s
Wall time: 1.19 s
CPU times: user 30.3 ms, sys: 0 ns, total: 30.3 ms
Wall time: 29.8 ms
(32, 96, 96, 3)
(32, 7)


In [24]:
print(img_size, type(img_size))
print(channel, type(channel))

96 <class 'int'>
3 <class 'int'>


In [25]:
# ====================================================================  
### Model Building      
# ==================================================================== 

In [26]:

#!pip install tensorflow==2.20.2 keras==3.8.0

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB2
print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)
print("tf.keras version:", tf.keras.__version__)
backbone = EfficientNetB2(
    input_shape=(img_size,img_size,channel),
    include_top=False,
    weights="imagenet"
)
model = tf.keras.Sequential([
    backbone,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(7,activation='softmax')
])

model.summary()

Keras version: 3.8.0
TensorFlow version: 2.20.0
tf.keras version: 3.8.0


KeyboardInterrupt: 

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001,
                                                 beta_1=0.9,
                                                  beta_2=0.999,
                                                  epsilon=1e-07),
            loss='categorical_crossentropy',
            metrics=['accuracy',
                     tf.keras.metrics.Precision(name='precision'),
                     tf.keras.metrics.Recall(name='recall')])


In [ ]:
#warm up the GPU
dummy_input = tf.random.normal([1,img_size,img_size,channel])
_ = model(dummy_input)

2025-10-16 09:50:49.893452: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


# ====================================================================  
### Training      
# ==================================================================== 

In [ ]:
history = model.fit(
    train_dataset,
    steps_per_epoch=int(6*(len(train_paths)//batch_size)),
    epochs=12,
    validation_data=val_dataset,
    class_weight=label_weights,
    validation_steps=int(len(val_paths)//batch_size)
)

Epoch 1/12


2025-10-16 09:51:24.628620: I external/local_xla/xla/service/service.cc:163] XLA service 0x79dbb80018b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-16 09:51:24.628678: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2025-10-16 09:51:26.691045: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-16 09:51:39.596733: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-16 09:51:41.488560: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spi

 744/5952 ━━━━━━━━━━━━━━━━━━━━ 5:30 63ms/step - accuracy: 0.3902 - loss: 2.5624 - precision: 0.6189 - recall: 0.1796

2025-10-16 09:54:32.590885: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng55{k2=6,k13=2,k14=2,k18=1,k22=0,k23=0} for conv (f32[12,96,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[12,16,48,48]{3,2,1,0}, f32[96,16,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2025-10-16 09:54:33.766721: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 2.175943803s
Trying algorithm eng55{k2=6,k13=2,k14=2,k18=1,k22=0,k23=0} for conv (f32[12,96,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[12,16,48,48]{3,2,1,0}, f32[96,16,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", bac

5952/5952 ━━━━━━━━━━━━━━━━━━━━ 1288s 190ms/step - accuracy: 0.5476 - loss: 1.9350 - precision: 0.7280 - recall: 0.3722 - val_accuracy: 0.6615 - val_loss: 0.9356 - val_precision: 0.7410 - val_recall: 0.5825
Epoch 2/12
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 1035s 174ms/step - accuracy: 0.7090 - loss: 1.2462 - precision: 0.8001 - recall: 0.6126 - val_accuracy: 0.6822 - val_loss: 0.9512 - val_precision: 0.7328 - val_recall: 0.6385
Epoch 3/12
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 1023s 172ms/step - accuracy: 0.7733 - loss: 0.9663 - precision: 0.8317 - recall: 0.7122 - val_accuracy: 0.6909 - val_loss: 0.9932 - val_precision: 0.7284 - val_recall: 0.6614
Epoch 4/12
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 1015s 171ms/step - accuracy: 0.8304 - loss: 0.7241 - precision: 0.8652 - recall: 0.7959 - val_accuracy: 0.6934 - val_loss: 1.1066 - val_precision: 0.7231 - val_recall: 0.6702
Epoch 5/12
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 997s 168ms/step - accuracy: 0.8716 - loss: 0.5548 - precision: 0.8934 - recall: 0.8513 - val_accuracy

In [ ]:
model.layers[0].trainable=False

In [ ]:
#callbacks
checkpoint=tf.keras.callbacks.ModelCheckpoint("best.weights.h5",
                                             verbose=1,
                                             save_best_only=True,
                                             save_weights_only=True)
early_stop=tf.keras.callbacks.EarlyStopping(patience=4)

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb2 (Functional)     │ (None, 3, 3, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       180,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,714,324 (90.46 MB)

 Trainable params: 181,255 (708.03 KB)

 Non-trainable params: 7,768,569 (29.63 MB)

 Optimizer params: 15,764,500 (60.14 MB)

In [ ]:
history =model.fit(
    train_dataset,
    steps_per_epoch=int(6*(len(train_paths)//batch_size)),
    epochs=10,
    callbacks=[checkpoint,early_stop],
    validation_data=val_dataset,
    validation_steps=int(len(val_paths)//batch_size),
    class_weight=label_weights
)

Epoch 1/10
 341/5952 ━━━━━━━━━━━━━━━━━━━━ 6:20 68ms/step - accuracy: 0.9558 - loss: 0.2093 - precision: 0.9613 - recall: 0.9508

5952/5952 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9551 - loss: 0.2009 - precision: 0.9592 - recall: 0.9515
Epoch 1: val_loss improved from inf to 1.67284, saving model to best.weights.h5
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 357s 60ms/step - accuracy: 0.9551 - loss: 0.2009 - precision: 0.9592 - recall: 0.9515 - val_accuracy: 0.6935 - val_loss: 1.6728 - val_precision: 0.7019 - val_recall: 0.6861
Epoch 2/10
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9580 - loss: 0.1911 - precision: 0.9622 - recall: 0.9548
Epoch 2: val_loss improved from 1.67284 to 1.63558, saving model to best.weights.h5
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 336s 56ms/step - accuracy: 0.9580 - loss: 0.1911 - precision: 0.9622 - recall: 0.9548 - val_accuracy: 0.6901 - val_loss: 1.6356 - val_precision: 0.7007 - val_recall: 0.6830
Epoch 3/10
5952/5952 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.9604 - loss: 0.1773 - precision: 0.9635 - recall: 0.9577
Epoch 3: val_loss did not improve from 1.63558
5952/5952 ━━━━

In [ ]:
backbone = EfficientNetB2(
    input_shape=(img_size,img_size,3),
    include_top=False,
    weights=None
)
mdl = tf.keras.Sequential([
    backbone,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dense(7,activation='softmax')
])
mdl.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001,
                                                 beta_1=0.9,
                                                  beta_2=0.999,
                                                  epsilon=1e-07),
                                                loss='categorical_crossentropy',
                                                metrics=['accuracy',
                                                tf.keras.metrics.Precision(name='precision'),
                                                tf.keras.metrics.Recall(name='recall')])
mdl.load_weights('best.weights.h5')

/home/lelandmesford/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 608 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
def decode_image_test(image,label):
    image = tf.io.read_file(image)
    image = tf.io.decode_jpeg(image,channels=3)
    image = tf.image.resize(image, [96,96],method='bilinear')
    return image,label

test_dataset=(tf.data.Dataset
              .from_tensor_slices((test_paths, test_labels))
              .map(decode_image_test)
              .batch(batch_size)
)

In [ ]:
test_dataset=test_dataset.shuffle(500)

In [ ]:
image,label=next(iter(test_dataset))
print(image.shape)
print(label.shape)

(32, 96, 96, 3)
(32, 7)


In [ ]:
loss, acc, prec, rec = mdl.evaluate(test_dataset)
metrics={'loss':loss,'accuracy':acc,'precision':prec,'recall':rec}
for k,v in metrics.items():
    print(f"{k}: {v}")

 73/225 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5937 - loss: 2.0989 - precision: 0.5987 - recall: 0.5805

2025-10-16 13:31:16.493696: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv (f32[10,528,6,6]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,528,6,6]{3,2,1,0}, f32[528,1,5,5]{3,2,1,0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, feature_group_count=528, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2025-10-16 13:31:16.854538: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.375567245s
Trying algorithm eng3{k11=2} for conv (f32[10,528,6,6]{3,2,1,0}, u8[0]{0}) custom-call(f32[10,528,6,6]{3,2,1,0}, f32[528,1,5,5]{3,2,1,0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, feature_group_count=528, custom_call_target="__cudnn$convFor

225/225 ━━━━━━━━━━━━━━━━━━━━ 32s 82ms/step - accuracy: 0.5920 - loss: 2.1309 - precision: 0.5999 - recall: 0.5804
loss: 2.120175361633301
accuracy: 0.5966843366622925
precision: 0.6071428656578064
recall: 0.5849819183349609


f1 best
accuracy: 0.5905544757843018
precision: 0.6223527789115906
recall: 0.5608804821968079

late night
accuracy: 0.5858177542686462
precision: 0.6047992706298828
recall: 0.568821370601654


previous best  
accuracy: 0.6053218245506287  
precision: 0.6987596154212952  
recall: 0.4944274127483368  

The course acheived  
acc 0.6348  
prec 0.7189  
rec 0.5553   
and stated that that was within 10% of the original competition  

This is no where close. There were two profiles with the Kabggle user's name and same data set title. Perhaps the other one is better:  
https://www.kaggle.com/datasets/sionehoghen/facial-expressions    

result: i ran the notebook with data1 minus the /facial-expresion/ folder and the result is not much different


In [ ]:

mdl.save('LATENIGHT_best_model.keras',save_format='tf')

In [ ]:
"""
import joblib 
from joblib import dump,load
import pickle

def save_object(obj,name):
    pickle_obj=open(f"{name}.pck","wb")
    pickle.dump(obj,pickle_obj)
    pickle_obj.close()
    """
    

'\nimport joblib \nfrom joblib import dump,load\nimport pickle\n\ndef save_object(obj,name):\n    pickle_obj=open(f"{name}.pck","wb")\n    pickle.dump(obj,pickle_obj)\n    pickle_obj.close()\n    '

In [ ]:
#save_object(le,'label_encoder')